In [ ]:
# Cell 1: Install system dependencies
!apt-get install -y bedtools samtools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
samtools is already the newest version (1.13-4).
bedtools is already the newest version (2.30.0+dfsg-2ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3: Download hg38 reference genome and index (skipped if exists)
%%bash
mkdir -p /content/drive/MyDrive/ML_Project/data
cd /content/drive/MyDrive/ML_Project/data

if [ ! -f hg38.fa ]; then
    echo "Downloading hg38.fa.gz ..."
    wget -q http://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
    echo "Decompressing ..."
    gunzip -k hg38.fa.gz
fi

if [ ! -f hg38.fa.fai ]; then
    echo "Indexing ..."
    samtools faidx hg38.fa
fi

In [ ]:
# Cell 4: Parse narrowPeak, center 101bp window on summit, filter boundaries
import pandas as pd
import os

BASE = '/content/drive/MyDrive/ML_Project/data'

# Config Datasets
DATASETS = {
    'SP1': 'sp1_raw_data.narrowPeak.bed',
    'SP2': 'sp2_raw_data.narrowPeak.bed',
    'SP4': 'sp4_raw_data.narrowPeak.bed',
}

COLS = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal', 'pval', 'qval', 'peak']

for name, filename in DATASETS.items():
    input_path = os.path.join(BASE, filename)
    output_path = os.path.join(BASE, f'{name.lower()}_centered_101bp.bed')

    if not os.path.exists(input_path):
        print(f"SKIP {name}: {filename} not found")
        continue

    df = pd.read_csv(input_path, sep='\t', header=None, names=COLS)
    df[['start', 'end', 'peak']] = df[['start', 'end', 'peak']].astype(int)

    df['summit'] = df['start'] + df['peak']
    df['new_start'] = df['summit'] - 50
    df['new_end'] = df['summit'] + 51

    initial = len(df)
    df = df[(df['new_start'] >= 0) & (df['new_end'] > df['new_start'])]

    df[['chrom', 'new_start', 'new_end']].to_csv(output_path, sep='\t', header=False, index=False)
    print(f"{name}: {initial} -> {len(df)} peaks -> {output_path}")

print("Done.")

SP1: 16043 -> 16043 peaks -> /content/drive/MyDrive/ML_Project/data/sp1_centered_101bp.bed
SP2: 11301 -> 11301 peaks -> /content/drive/MyDrive/ML_Project/data/sp2_centered_101bp.bed
SP4: 23574 -> 23574 peaks -> /content/drive/MyDrive/ML_Project/data/sp4_centered_101bp.bed
Done.


In [ ]:
# Cell 5: Extract chromosome sizes from FASTA index
%%bash
cd /content/drive/MyDrive/ML_Project/data
cut -f 1,2 hg38.fa.fai > hg38.chrom.sizes
echo "Generated hg38.chrom.sizes"

Generated hg38.chrom.sizes


In [ ]:
# Cell 6: Generate positive/negative FASTA + clean for SP1, SP2, SP4
import os
import subprocess

BASE = "/content/drive/MyDrive/ML_Project/data"
PROTEINS = ['sp1', 'sp2', 'sp4']

# PHASE 1: bedtools getfasta + shuffle (bash)
for protein in PROTEINS:
    bed_in = f"{BASE}/{protein}_centered_101bp.bed"
    fa_pos = f"{BASE}/{protein}_positive_101bp.fasta"
    bed_neg = f"{BASE}/{protein}_negative_101bp.bed"
    fa_neg = f"{BASE}/{protein}_negative_101bp.fasta"

    if not os.path.exists(bed_in):
        print(f"SKIP {protein}: {bed_in} not found")
        continue

    print(f"\n--- {protein.upper()} ---")

    # Extract positive sequences
    subprocess.run([
        'bedtools', 'getfasta',
        '-fi', f'{BASE}/hg38.fa',
        '-bed', bed_in,
        '-fo', fa_pos
    ], check=True)
    print(f"  Positive FASTA saved.")

    # Generate negative coordinates
    with open(bed_neg, 'w') as fout:
        subprocess.run([
            'bedtools', 'shuffle',
            '-i', bed_in,
            '-g', f'{BASE}/hg38.chrom.sizes',
            '-excl', bed_in,
            '-noOverlapping',
            '-seed', '42'
        ], stdout=fout, check=True)
    print(f"  Negative BED saved.")

    # Extract negative sequences
    subprocess.run([
        'bedtools', 'getfasta',
        '-fi', f'{BASE}/hg38.fa',
        '-bed', bed_neg,
        '-fo', fa_neg
    ], check=True)
    print(f"  Negative FASTA saved.")

# PHASE 2: Clean FASTA (remove N, uppercase)
def clean_fasta(input_path, output_path):
    total, valid, discarded = 0, 0, 0
    with open(input_path, 'r') as fin, open(output_path, 'w') as fout:
        header, seq_parts = None, []
        for line in fin:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    total += 1
                    seq = ''.join(seq_parts).upper()
                    if 'N' not in seq:
                        fout.write(f"{header}\n{seq}\n")
                        valid += 1
                    else:
                        discarded += 1
                header, seq_parts = line, []
            else:
                seq_parts.append(line)
        if header is not None:
            total += 1
            seq = ''.join(seq_parts).upper()
            if 'N' not in seq:
                fout.write(f"{header}\n{seq}\n")
                valid += 1
            else:
                discarded += 1
    return total, valid, discarded

# PHASE 3: Execute cleaning + summary
print("\n" + "="*50)
print("CLEANING PIPELINE")
print("="*50)

summary = {}
for protein in PROTEINS:
    pos_raw = f"{BASE}/{protein}_positive_101bp.fasta"
    neg_raw = f"{BASE}/{protein}_negative_101bp.fasta"
    pos_clean = f"{BASE}/{protein}_positive_101bp_clean.fasta"
    neg_clean = f"{BASE}/{protein}_negative_101bp_clean.fasta"

    if not os.path.exists(pos_raw):
        continue

    print(f"\n--- {protein.upper()} ---")
    _, v_pos, _ = clean_fasta(pos_raw, pos_clean)
    _, v_neg, _ = clean_fasta(neg_raw, neg_clean)

    summary[protein] = (v_pos, v_neg)
    status = "BALANCED" if v_pos == v_neg else f"IMBALANCE: {abs(v_pos-v_neg)}"
    print(f"  Result: pos={v_pos}, neg={v_neg} -> {status}")

print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
total_pos = sum(v[0] for v in summary.values())
total_neg = sum(v[1] for v in summary.values())
for protein, (pos, neg) in summary.items():
    print(f"  {protein.upper()}: pos={pos}, neg={neg}")
print(f"  TOTAL: pos={total_pos}, neg={total_neg}")
print(f"  Combined dataset: {total_pos + total_neg} sequences")


--- SP1 ---
  Positive FASTA saved.
  Negative BED saved.
  Negative FASTA saved.

--- SP2 ---
  Positive FASTA saved.
  Negative BED saved.
  Negative FASTA saved.

--- SP4 ---
  Positive FASTA saved.
  Negative BED saved.
  Negative FASTA saved.

CLEANING PIPELINE

--- SP1 ---
  Result: pos=16043, neg=15210 -> IMBALANCE: 833

--- SP2 ---
  Result: pos=11301, neg=10727 -> IMBALANCE: 574

--- SP4 ---
  Result: pos=23574, neg=22386 -> IMBALANCE: 1188

FINAL SUMMARY
  SP1: pos=16043, neg=15210
  SP2: pos=11301, neg=10727
  SP4: pos=23574, neg=22386
  TOTAL: pos=50918, neg=48323
  Combined dataset: 99241 sequences
